In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


In [2]:
# Cargar los datasets
df_isss = pd.read_excel("BASE DEL ISSS JUNIO 2024 CIBERINTELIGENCIASV (2) (2).xlsx", dtype=str)
df_altas1 = pd.read_excel("altas_pospago_res.xlsx", dtype=str)
df_altas2 = pd.read_excel("altas_pospago_res2.xlsx", dtype=str)
df_parque = pd.read_excel("parque_pospago_res.xlsx", dtype=str)

print("Archivos cargados correctamente.")


Archivos cargados correctamente.


In [3]:
def limpiar_texto(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].str.strip().replace("", np.nan)
    return df

df_isss = limpiar_texto(df_isss)
df_altas1 = limpiar_texto(df_altas1)
df_altas2 = limpiar_texto(df_altas2)
df_parque = limpiar_texto(df_parque)

print("Limpieza general aplicada.")


Limpieza general aplicada.


In [4]:
df_altas = pd.concat([df_altas1, df_altas2], ignore_index=True)
df_altas.drop_duplicates(inplace=True)

print(f"Registros totales en altas pospago: {len(df_altas)}")


Registros totales en altas pospago: 80047


In [5]:
df_altas.columns.tolist()


['TELEFONO',
 'FECHA_ALTA',
 'TIPO_ALTA',
 'SAD_ANALISTA',
 'MOTIVO_OBSERVACION',
 'SAD_SUBTIPOLOGIA',
 'ID_SOLICITUD',
 'ID_REDSHIFT',
 'FECHA_SOLICITUD',
 'PRODUCTO',
 'TIPO_ALTA_BOT',
 'OFERTA_MODALIDAD',
 'OFERTA_PLAN',
 'OFERTA_VIGENCIA',
 'OFERTA_GARANTIA',
 'OFERTA_MODELO',
 'VENDEDOR_CHATID',
 'VENDEDOR_NOMBRE',
 'VENDEDOR_CANAL',
 'VENDEDOR_COMERCIALIZADOR',
 'VENDEDOR_DEPARTAMENTO',
 'VENDEDOR_PUNTOVENTA',
 'VENDEDOR_ID_PREPAGO',
 'VENDEDOR_ID_POSPAGO',
 'CLIENTE_NOMBRE',
 'CLIENTE_DUI',
 'CUENTA_ARBOR',
 'CLIENTE_GENERO',
 'CLIENTE_FECHANAC',
 'EDAD',
 'SALDO',
 'DIAS_MORA',
 'RANGO_MORA',
 'DIFERIDO',
 'CUOTAS_PENDIENTES',
 'ESTADO_ARBOR',
 'SCORE_EXT',
 'SCORE_INT',
 'RIESGO',
 '%TARJ_CREDIT',
 'RANGO_TARJETA',
 'PORCENTAJE_TARJ_PTS',
 'LIMITE',
 'LIMITE_INTERNET',
 'OFERTA_SCORE',
 'INGRESO_MONTO',
 'INGRESO_PTS',
 'DEUDA_PORCENTAJE',
 'DEUDA_PORCENTAJE_PTS',
 'VECESMORA_U6MESES_PTS',
 'VECESMORA_MAYOR29DIAS_PTS',
 'MESES_ANTIG_LABORAL',
 'MESES_ANTIG_LABORAL_PTS',
 'SEGM

In [6]:
df_altas[
    ["INGRESO_MONTO",
     "DEUDA_PORCENTAJE",
     "%TARJ_CREDIT",
     "SCORE_EXT",
     "VECESMORA_U6MESES_PTS"]
].isna().all(axis=1).sum()



4725

In [7]:
df_altas = df_altas[
    ~df_altas[
        ["INGRESO_MONTO",
         "DEUDA_PORCENTAJE",
         "%TARJ_CREDIT",
         "SCORE_EXT",
         "VECESMORA_U6MESES_PTS"]
    ].isna().all(axis=1)
]


In [8]:
def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.str.replace(r"[^\d.-]", "", regex=True),
        errors="coerce"
    )

numeric_cols = [
    "INGRESO_MONTO","DEUDA_PORCENTAJE","SALDO","DIAS_MORA",
    "LIMITE","LIMITE_INTERNET","%TARJ_CREDIT","EDAD"
]

for col in numeric_cols:
    if col in df_altas.columns:
        df_altas[col] = limpiar_numerico(df_altas[col])

print("Columnas numéricas convertidas.")


Columnas numéricas convertidas.


In [9]:
date_cols = [
    "FECHA_ALTA", "FECHA_SOLICITUD",
    "CLIENTE_FECHANAC", "FEC_ULTIMA_RENO"
]

for col in date_cols:
    if col in df_altas.columns:
        df_altas[col] = pd.to_datetime(df_altas[col], errors="coerce")

print("Fechas convertidas.")


Fechas convertidas.


In [ ]:
# Normalizar DUI en df_isss
df_isss["DUI_clean"] = (
    df_isss["DUI"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)  
    .str.zfill(9)                        
)

# Normalizar DUI en df_altas
df_altas["CLIENTE_DUI_clean"] = (
    df_altas["CLIENTE_DUI"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(9)
)


In [12]:
df_model = pd.DataFrame()

df_model["DUI_clean"] = df_altas["CLIENTE_DUI_clean"]
df_model["edad"] = df_altas["EDAD"]
df_model["ingresos_declarados"] = df_altas["INGRESO_MONTO"]
df_model["nivel_endeudamiento"] = df_altas["DEUDA_PORCENTAJE"] / 100
df_model["utilizacion_tarjetas"] = df_altas["%TARJ_CREDIT"] / 100
df_model["score_buro"] = df_altas["SCORE_EXT"]
df_model["morosidad_prev"] = df_altas["VECESMORA_U6MESES_PTS"]
df_model["historial_empresa"] = df_altas["FEC_ULTIMA_RENO"].notna().astype(int)


df_model.head()


,DUI_clean,edad,ingresos_declarados,nivel_endeudamiento,utilizacion_tarjetas,score_buro,morosidad_prev,historial_empresa
0,036787389,39.0,1050.0,0.1711,0.50,513,-5,0
3,057164229,27.0,NaN,NaN,0.00,0,NaN,0
4,049908245,31.0,425.0,0.0000,0.05,407,0,0
5,020495348,57.0,NaN,NaN,0.00,0,NaN,0
6,026590837,64.0,NaN,NaN,0.00,0,NaN,0


In [13]:
salario_map = (
    df_isss
    .dropna(subset=["DUI_clean", "SALARIO"])
    .drop_duplicates("DUI_clean")
    .set_index("DUI_clean")["SALARIO"]
)


In [14]:
# Definir qué consideramos "vacío"
mask_vacio = (
    df_model["ingresos_declarados"].isna() |
    (df_model["ingresos_declarados"].astype(str).str.strip() == "")
)

# Aplicar el mapeo solo a esos casos
df_model.loc[mask_vacio, "ingresos_declarados"] = (
    df_model.loc[mask_vacio, "DUI_clean"].map(salario_map)
)


C:\Users\alexg\AppData\Local\Temp\ipykernel_6052\4245972287.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['1282.97' nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_model.loc[mask_vacio, "ingresos_declarados"] = (


In [17]:
columnas_a_reemplazar = [
    "ingresos_declarados",
    "nivel_endeudamiento",
    "utilizacion_tarjetas",
    "score_buro",
    "morosidad_prev"
]

# Convertir valores vacíos a NaN primero
df_model[columnas_a_reemplazar] = df_model[columnas_a_reemplazar].replace("", np.nan)

# Reemplazar NaN por 0
df_model[columnas_a_reemplazar] = df_model[columnas_a_reemplazar].fillna(0)

# Asegurar que sean numéricos
df_model[columnas_a_reemplazar] = df_model[columnas_a_reemplazar].apply(pd.to_numeric, errors="coerce").fillna(0)


In [26]:
df_model = df_model[~df_model["edad"].isna()]


In [27]:
df_model.to_csv("dataset_final_modelo.csv", index=False, encoding="utf-8")

print("Dataset final exportado como 'dataset_final_modelo.csv'")


Dataset final exportado como 'dataset_final_modelo.csv'
